# Notebook 02 — Raw → Clean

**Scope:** full dataset, both taxi types, Jan 2024–May 2026 (29 months each).

This notebook runs the actual Raw → Clean pipeline implemented in `src/cleaning.py`
(the same code `python -m src.cleaning all` runs), then surfaces:

- **Section 2.1**'s exact row-removal counts (`trip_distance <= 0`, `fare_amount < 0`,
  `trip_duration_minutes <= 0` or `> 1440`), combined Yellow+Green
- The **Clean** row of **Table 2** (pipeline shape)

Every removal rule this stage applies is logged to a ledger with an exact count and
percentage, computed from the data — this notebook prints the *full* ledger (not just
the three headline rules highlighted below) so nothing here is hidden or cherry-picked.
Rows are removed only for a specific, documented reason; anything merely unusual is
flagged (`is_extreme_trip_distance`, `is_extreme_trip_duration`), never silently dropped.

## 1. Setup

In [1]:
from pyspark.sql import SparkSession

from src.cleaning import clean_taxi_type, write_clean_dataset_per_month, write_reports

# Same config as `python -m src.cleaning all` -- Yellow alone is ~109M rows across 29
# files read together, plus a dropDuplicates() shuffle over ~20 columns, which OOMs
# on the default ~1GB driver heap.
spark = (
    SparkSession.builder
    .appName("nyc-taxi-raw-to-clean")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.memory", "16g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")

2026-08-28 16:02:03.680 | INFO     | src.config:<module>:26 - PROJ_ROOT path is: /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/28 06:02:04 WARN Utils: Your hostname, Debaos-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.2 instead (on interface en0)
26/08/28 06:02:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/28 06:02:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


## 2. Clean Yellow Taxi

Loads all 29 Yellow Raw month-files, profiles them, and applies every removal rule in
`src/clean_rules.py` / `src/cleaning.py`.

In [2]:
yellow_result = clean_taxi_type(spark, "yellow")
print(
    f"Yellow: {yellow_result['clean_row_count']:,} / {yellow_result['raw_row_count']:,} "
    f"rows retained ({100 * yellow_result['clean_row_count'] / yellow_result['raw_row_count']:.2f}%)"
)

2026-08-28 06:02:08.344 | INFO     | src.cleaning:clean_taxi_type:377 - [yellow] loaded 108,891,604 Raw rows


2026-08-28 06:02:57.597 | INFO     | src.cleaning:apply_removal_rule:292 - [yellow] removed 2,756,007 rows: trip_distance <= 0


2026-08-28 06:02:58.903 | INFO     | src.cleaning:apply_removal_rule:292 - [yellow] removed 3,384,298 rows: fare_amount < 0


2026-08-28 06:02:59.839 | INFO     | src.cleaning:apply_removal_rule:292 - [yellow] removed 764,283 rows: trip_duration_minutes <= 0 or > 1440 (24h)


2026-08-28 06:03:53.880 | INFO     | src.cleaning:apply_duplicate_removal:325 - [yellow] removed 2 duplicate rows


2026-08-28 06:04:00.789 | INFO     | src.cleaning:apply_removal_rule:292 - [yellow] removed 67 rows: pickup_datetime outside study period [2024-01-01 00:00:00, 2026-06-01 00:00:00)


2026-08-28 06:07:00.997 | INFO     | src.cleaning:add_extreme_value_flags:349 - [yellow] extreme-value flag thresholds (Q3 + 3.7942*IQR, N=101,986,947): trip_distance > 13.267327667676824, trip_duration_minutes > 72.12297988281799
2026-08-28 06:07:01.018 | SUCCESS  | src.cleaning:clean_taxi_type:519 - [yellow] Clean: 101,986,947 / 108,891,604 rows retained (93.66%)
Yellow: 101,986,947 / 108,891,604 rows retained (93.66%)


## 3. Clean Green Taxi

Same process, all 29 Green Raw month-files.

In [3]:
green_result = clean_taxi_type(spark, "green")
print(
    f"Green: {green_result['clean_row_count']:,} / {green_result['raw_row_count']:,} "
    f"rows retained ({100 * green_result['clean_row_count'] / green_result['raw_row_count']:.2f}%)"
)

2026-08-28 06:07:01.163 | INFO     | src.cleaning:clean_taxi_type:377 - [green] loaded 1,462,605 Raw rows


2026-08-28 06:07:07.212 | INFO     | src.cleaning:apply_removal_rule:292 - [green] removed 66,282 rows: trip_distance <= 0


2026-08-28 06:07:08.018 | INFO     | src.cleaning:apply_removal_rule:292 - [green] removed 2,686 rows: fare_amount < 0


2026-08-28 06:07:08.412 | INFO     | src.cleaning:apply_removal_rule:292 - [green] removed 1,552 rows: trip_duration_minutes <= 0 or > 1440 (24h)


2026-08-28 06:07:10.811 | INFO     | src.cleaning:apply_removal_rule:292 - [green] removed 10 rows: pickup_datetime outside study period [2024-01-01 00:00:00, 2026-06-01 00:00:00)


2026-08-28 06:07:14.938 | INFO     | src.cleaning:add_extreme_value_flags:349 - [green] extreme-value flag thresholds (Q3 + 3.2612*IQR, N=1,392,075): trip_distance > 10.877604452149203, trip_duration_minutes > 54.43791187260208
2026-08-28 06:07:14.948 | SUCCESS  | src.cleaning:clean_taxi_type:519 - [green] Clean: 1,392,075 / 1,462,605 rows retained (95.18%)
Green: 1,392,075 / 1,462,605 rows retained (95.18%)


## 4. Write Clean dataset + data-quality reports

Writes the unified Clean taxi dataset to `data/clean/taxi/` (one flat parquet file per
taxi type per month) and the removal ledger / profiles to
`reports/data_quality/clean/`.

In [4]:
results = [yellow_result, green_result]
for r in results:
    write_clean_dataset_per_month(r["clean_df"], r["taxi_type"])

combined_clean_row_count = sum(r["clean_row_count"] for r in results)
write_reports(results, combined_clean_row_count)
print("Clean dataset + data-quality reports written.")

2026-08-28 06:11:16.104 | INFO     | src.cleaning:write_clean_dataset_per_month:589 - [yellow] wrote 29 monthly Clean file(s) to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/clean/taxi


2026-08-28 06:11:20.760 | INFO     | src.cleaning:write_clean_dataset_per_month:589 - [green] wrote 29 monthly Clean file(s) to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/clean/taxi
2026-08-28 06:11:20.781 | INFO     | src.cleaning:write_reports:638 - Clean data-quality reports written to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/reports/data_quality/clean
Clean dataset + data-quality reports written.


## 5. Full row-removal ledger (combined Yellow+Green)

Every rule applied at this stage, in order, with the exact count and percentage of rows
it removed — combined across both taxi types. Percentages here are against the
**combined** raw row count (110,354,209).

In [5]:
import pandas as pd

combined_raw_row_count = sum(r["raw_row_count"] for r in results)

all_ledger_rows = [row for r in results for row in r["ledger"]]
ledger_df = pd.DataFrame(all_ledger_rows)

combined_ledger = (
    ledger_df.groupby(["stage", "rule", "reason"], sort=False)["records_removed"]
    .sum()
    .reset_index()
)
combined_ledger["pct_of_combined_raw"] = (
    100 * combined_ledger["records_removed"] / combined_raw_row_count
).round(4)

pd.set_option("display.max_colwidth", None)
combined_ledger

,stage,rule,reason,records_removed,pct_of_combined_raw
0,essential_missing,pickup_datetime IS NULL,pickup_datetime is essential for zone-hour pickup-demand analysis,0,0.0000
1,essential_missing,pu_location_id IS NULL,pu_location_id is essential for zone-hour pickup-demand analysis,0,0.0000
2,invalid,"pu_location_id outside valid TLC zone range [1, 265]",pu_location_id is essential and must reference a real/documented TLC taxi zone,0,0.0000
3,invalid,"do_location_id outside valid TLC zone range [1, 265]",dropoff location must reference a real/documented TLC taxi zone (per user cleaning policy),0,0.0000
4,invalid,trip_distance <= 0,negative distance is physically impossible; zero-distance trips dropped per user cleaning policy,2822289,2.5575
5,invalid,passenger_count < 0,a physically impossible passenger count,0,0.0000
6,invalid,fare_amount < 0,negative fares dropped per user cleaning policy,3386984,3.0692
7,invalid,trip_duration_minutes <= 0 or > 1440 (24h),"non-positive or >24h trips dropped per user cleaning policy (supersedes the narrower dropoff < pickup check, which this subsumes)",765835,0.6940
8,invalid,exact duplicate business-column record,identical trip-defining values (all Clean business fields) already represented by another row,2,0.0000
9,study_period,"pickup_datetime outside study period [2024-01-01 00:00:00, 2026-06-01 00:00:00)",out of the established study period; not a data-validity issue,77,0.0001


## 6. Section 2.1 — the three headline removal rules

The three rules with the largest, individually quantified impact.

In [6]:
headline_rules = {
    "trip_distance <= 0": "trip_distance <= 0",
    "fare_amount < 0": "fare_amount < 0",
    "trip_duration_minutes <= 0 or > 1440 (24h)": "trip_duration_minutes <= 0 or > 1440",
}

print("Section 2.1 -- combined Yellow+Green, all 29 months")
for ledger_rule, report_label in headline_rules.items():
    row = combined_ledger.loc[combined_ledger["rule"] == ledger_rule].iloc[0]
    print(
        f"  {report_label}: {int(row['records_removed']):,} rows, "
        f"{row['pct_of_combined_raw']:.2f}%"
    )

Section 2.1 -- combined Yellow+Green, all 29 months
  trip_distance <= 0: 2,822,289 rows, 2.56%
  fare_amount < 0: 3,386,984 rows, 3.07%
  trip_duration_minutes <= 0 or > 1440: 765,835 rows, 0.69%


## 7. Table 2 — pipeline shape, Clean row

The Clean stage's row counts, and the overall percentage of rows removed by the
cleaning rules above.

In [7]:
yellow_clean = yellow_result["clean_row_count"]
green_clean = green_result["clean_row_count"]
combined_clean = yellow_clean + green_clean
pct_removed = 100 * (combined_raw_row_count - combined_clean) / combined_raw_row_count

print("Table 2 -- Clean stage")
print(f"  Yellow rows:   {yellow_clean:,}")
print(f"  Green rows:    {green_clean:,}")
print(f"  Combined rows: {combined_clean:,}")
print(f"  Clean removes {pct_removed:.2f}% of rows combined")

Table 2 -- Clean stage
  Yellow rows:   101,986,947
  Green rows:    1,392,075
  Combined rows: 103,379,022
  Clean removes 6.32% of rows combined


In [8]:
spark.stop()

## Notes

- The full ledger above (Section 5) includes several rules beyond the three headline
  ones highlighted in Section 6 — `pickup_datetime`/`pu_location_id` null, out-of-range
  `pu_location_id`/`do_location_id`, exact duplicates, rows outside the study period, and
  `passenger_count < 0` — included here for full transparency about what the pipeline
  actually removes, even though only three rules are large enough to warrant individual
  quantification.
- Output: one flat parquet file per taxi type per month under `data/clean/taxi/`, plus
  the removal ledger and per-taxi-type profiles under `reports/data_quality/clean/`.
- Raw → Clean row counts here feed Table 2; Section 2.2's external-join-coverage
  numbers and the rest of Table 2's Processed row are computed in notebooks 03 and 04.